In [ ]:
import pandas as pd 
import plotly.express as px
import random 


#definindo bicho
bicho = {
    "data_nascimento":0,
    "vivo":True,
    "expectativa_de_vida":3,
    "anos_de_vida":None,
    "data_morte":None,
    "dieta":None 
    
}


#definindo floresta
quantidade_bichos_inicial = 10
lista_bichos = []
dieta_lista = ["herbivoro","carnivoro"]

for _ in range(quantidade_bichos_inicial):
    bicho_inicial = bicho.copy()
    dieta = random.choices(dieta_lista,weights=[1,1])[0]
    anos_de_vida = round(random.gauss(mu=bicho["expectativa_de_vida"], sigma=1.0))
    bicho_inicial['anos_de_vida'] = anos_de_vida
    bicho_inicial["dieta"] = dieta
    lista_bichos.append(bicho_inicial)

df_estado_inicial = pd.DataFrame(lista_bichos)
df_inicial_dieta = df_estado_inicial.groupby("dieta")["vivo"].count()

print("distribuicao inicial dieta:")
print(df_inicial_dieta)

#loop dos anos
max_anos = 30

for ano_atual in range(1,max_anos+1):
    novos_bichos = []
    lista_bichos_vivos = [b for b in lista_bichos if b["vivo"]]
    print(f"ano:{ano_atual}: tamanho bichos vivos {len(lista_bichos_vivos)}")
    for bicho in lista_bichos_vivos:

        idade = ano_atual - bicho["data_nascimento"]

        anos_de_vida = bicho['anos_de_vida']
        if idade >= anos_de_vida:
            bicho['vivo'] = False
            bicho['data_morte'] = ano_atual

        else:
            dieta = bicho["dieta"]
            if dieta == "carnivoro":
                presa = random.choice(lista_bichos_vivos)
                
                dieta_presa = presa["dieta"]

                if dieta_presa == "carnivoro":
                    continue
                else:
                    presa["vivo"] = False
                    presa["data_morte"] = ano_atual
                    novo_bicho = bicho.copy()
                    novo_bicho["data_nascimento"] = ano_atual
                    novos_bichos.append(novo_bicho)
            else:  # é herbivoro   
                predador = random.choice(lista_bichos_vivos)
                dieta_predador = predador['dieta']
                if dieta_predador == 'herbivoro':

                    novo_bicho = bicho.copy()
                    novo_bicho["data_nascimento"] = ano_atual
                    novos_bichos.append(novo_bicho)
                
                else:#predador carnivoro
                    bicho['vivo'] = False
                    bicho['data_morte'] = ano_atual


            
    lista_bichos.extend(novos_bichos)

df = pd.DataFrame(lista_bichos)

#adicionando coluna de idade 

df["idade"] = df["data_morte"] - df["data_nascimento"]
df.loc[df["idade"].isna(),"idade"] = max_anos - df.loc[df["idade"].isna(),"data_nascimento"] 

#analises 
def contagem_total_grafico(df):

    # conta quantos bichos nasceram em cada data
    contagem = (
        df
        .groupby("data_nascimento")
        .size()
        .reset_index(name="qtd_bichos")          # renomeia a coluna de contagem
        .sort_values("data_nascimento")         # garante ordenação
    )

    # soma acumulada
    contagem["total_acumulado"] = contagem["qtd_bichos"].cumsum()

    fig = px.line(
        contagem,
        x="data_nascimento",
        y="total_acumulado",
        title="contagem_de_bixos"
    )

    fig.show()



df["idade"].describe()

distribuicao inicial dieta:
dieta
carnivoro    3
herbivoro    7
Name: vivo, dtype: int64
ano:1: tamanho bichos vivos 10
ano:2: tamanho bichos vivos 11
ano:3: tamanho bichos vivos 11
ano:4: tamanho bichos vivos 9
ano:5: tamanho bichos vivos 10
ano:6: tamanho bichos vivos 16
ano:7: tamanho bichos vivos 28
ano:8: tamanho bichos vivos 50
ano:9: tamanho bichos vivos 88
ano:10: tamanho bichos vivos 156
ano:11: tamanho bichos vivos 278
ano:12: tamanho bichos vivos 496
ano:13: tamanho bichos vivos 888
ano:14: tamanho bichos vivos 1594
ano:15: tamanho bichos vivos 2868
ano:16: tamanho bichos vivos 5172
ano:17: tamanho bichos vivos 9346
ano:18: tamanho bichos vivos 16920
ano:19: tamanho bichos vivos 30684
ano:20: tamanho bichos vivos 55730
ano:21: tamanho bichos vivos 101360
ano:22: tamanho bichos vivos 184580
ano:23: tamanho bichos vivos 336502
ano:24: tamanho bichos vivos 614080
ano:25: tamanho bichos vivos 1121632
ano:26: tamanho bichos vivos 2050322
ano:27: tamanho bichos vivos 3750612
ano:2

KeyboardInterrupt: 

In [ ]:
df_vivos = df.loc[df["data_morte"].isna()].copy()


resumo_idade_total = df["idade"].describe().to_frame("total")
resumo_idade_herbivoro = df.loc[df["dieta"] == "herbivoro", "idade"].describe().to_frame("herbivoro")
resumo_idade_carnivoro = df.loc[df["dieta"] == "carnivoro", "idade"].describe().to_frame("carnivoro")

resumo_idade_final = pd.concat([resumo_idade_total, resumo_idade_herbivoro, resumo_idade_carnivoro],axis=1)


resumo_idade_total_vivo = df_vivos["idade"].describe().to_frame("total_vivo")
resumo_idade_herbivoro_vivo = df_vivos.loc[df_vivos["dieta"] == "herbivoro", "idade"].describe().to_frame("herbivoro_vivo")
resumo_idade_carnivoro_vivo = df_vivos.loc[df_vivos["dieta"] == "carnivoro", "idade"].describe().to_frame("carnivoro_vivo")

resumo_idade_final_vivo = pd.concat([resumo_idade_total_vivo, resumo_idade_herbivoro_vivo, resumo_idade_carnivoro_vivo],axis=1)

resumo_idade = pd.concat([resumo_idade_final,resumo_idade_final_vivo],axis=1)
resumo_idade

,total,herbivoro,carnivoro,total_vivo,herbivoro_vivo,carnivoro_vivo
count,35.000000,14.000000,21.000000,11.000000,2.000000,9.000000
mean,1.685714,1.000000,2.142857,1.000000,0.500000,1.111111
std,1.131668,0.784465,1.108409,0.774597,0.707107,0.781736
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.250000,1.000000,0.500000,0.250000,1.000000
50%,2.000000,1.000000,2.000000,1.000000,0.500000,1.000000
75%,3.000000,1.750000,3.000000,1.500000,0.750000,2.000000
max,4.000000,2.000000,4.000000,2.000000,1.000000,2.000000


In [ ]:
def populacao_ano(df):


    nascimento_ano = df.groupby(["data_nascimento","dieta"]).agg(nascimentos=("dieta","count")).reset_index()
    morte_ano = df.groupby(["data_morte","dieta"]).agg(mortes=("dieta","count")).reset_index()
    df_censo = pd.merge(nascimento_ano,morte_ano,how="outer",left_on=["data_nascimento","dieta"],right_on=["data_morte","dieta"])

    df_censo = df_censo.rename(columns={"data_nascimento":"ano"})
    df_censo.loc[df_censo["ano"].isna(),"ano"] = df_censo.loc[df_censo["ano"].isna(),"data_morte"] 

    df_censo.drop(columns=["data_morte"],inplace=True)

    start_year = int(df_censo['ano'].min())
    end_year = int(df_censo['ano'].max())

    all_years = range(start_year, end_year + 1)
    all_categories = df_censo['dieta'].unique()

    new_index = pd.MultiIndex.from_product(
        [all_categories, all_years], 
        names=['dieta', 'ano']
    )

    df_censo = (
        df_censo.set_index(['dieta', 'ano'])
        .reindex(new_index)
        .reset_index()
    )
    df_censo = df_censo.fillna(0,inplace=False)

    ordem_das_colunas = ["ano","dieta","nascimentos","mortes"]
    df_censo = df_censo[ordem_das_colunas]
    df_censo = df_censo.sort_values(by="ano")
    df_censo["natalidade"] = df_censo["nascimentos"] - df_censo["mortes"]

    df_censo["populacao"] = df_censo.groupby(["dieta"])["natalidade"].transform("cumsum")

    fig = px.line(
        df_censo,
        x="ano",
        y="populacao",
        color= "dieta",
        title="populacao"
    )

    fig.show()

    return df_censo


    


populacao_ano(df)


,ano,dieta,nascimentos,mortes,natalidade,populacao
0,0,carnivoro,9.0,0.0,9.0,9.0
6,0,herbivoro,1.0,0.0,1.0,1.0
7,1,herbivoro,1.0,2.0,-1.0,0.0
1,1,carnivoro,1.0,0.0,1.0,10.0
2,2,carnivoro,0.0,4.0,-4.0,6.0
8,2,herbivoro,0.0,0.0,0.0,0.0
9,3,herbivoro,0.0,0.0,0.0,0.0
3,3,carnivoro,0.0,3.0,-3.0,3.0
10,4,herbivoro,0.0,0.0,0.0,0.0
4,4,carnivoro,0.0,1.0,-1.0,2.0


In [ ]:

df.loc[df["idade"]<0]